## Rule-based Tense Semantics error generation

This notebook is used for Tense Semantics error generation using heuristic code as opposed to PPO training.

In [ ]:
!pip install pyinflect

In [ ]:
! pip install transformers datasets evaluate

In [27]:
from transformers import pipeline
from transformers import AutoTokenizer
import spacy
import pyinflect

In [ ]:
generator = pipeline("text-generation", model=FINETUNED_DISTILGPT_PATH) ## model after initial finetuning, not PPO-trained, path removed for anonymity

In [13]:
generator("It can be seen", num_return_sequences=4, return_full_text=True, do_sample=True, temperature=1)

[{'generated_text': 'It can be seen that the majority of the population of Yemen is older than 60 years and the quantity of children will grow gradually. The number of boys and girls will grow steadily and boys will have less opportunities to do sports. It is not surprising that'},
 {'generated_text': 'It can be seen that the highest amount of overweight people was in Italy, where it was in 2000. It is also seen that obesity is a big part of the population, which is about 13 times more then obesity in Indonesia. It is also seen'},
 {'generated_text': 'It can be seen that the biggest percentage of children without access to primary school education was in Africa in 2000 and it is the lowest percentage in the whole world. In other countries children are more and more stable and the amount of children without access to primary'},
 {'generated_text': 'It can be seen, that the number of girls has the lowest amount of education in North Asia and Worldwide. In the same period the percentage o

In [14]:
generator("It can be seen", num_return_sequences=4, return_full_text=True, do_sample=True, temperature=1)

[{'generated_text': 'It can be seen from the graph that there are two most popular regions for this kind of information: USA and Sweden also have the most popular population share among people in Sweden. On the contrary, this data shows in the United Kingdom there are more people'},
 {'generated_text': 'It can be seen from the chart, that in 1999 the amount of children without access to primary education was about twice more in this country than in 2000. The percentage of children without access to primary education is bigger in North Asia compared with South Asia.'},
 {'generated_text': 'It can be seen from the chart that in 2011 amount of adults aged 65 was increasing. It had grown more steadily in the period of 2000 and then on the year 2000 started to grow up. It reached its peak in 2013 and reached it level of'},
 {'generated_text': 'It can be seen from the graph chart the number of children without access to primary education has also fallen from 3,3 per cent to 2,1 per cent, and

In [ ]:
generator = pipeline("text-generation", model=PPO_TRAINED_TEMSEM_GENERATOR_PATH) ## removed for anonymity

Temperature influence on generation diversity

In [70]:
generator("It can be seen", num_return_sequences=10, return_full_text=True, do_sample=True, temperature=0.5)

[{'generated_text': 'It can be seen that in 2012 the percentage of unemployment rate was lower than in 2012, but in 2013 the number of unemployment rate was lower than in 2012. In other words, in 2013 the number of unemployment rate was higher than in 2012, but'},
 {'generated_text': 'It can be seen that in 1985 the percentage of boys and girls was higher than girls. In 1995 the percentage of girls in different schools was slightly higher than the percentage of boys. In 2006 the proportion of boys was higher than boys. In 2005 the'},
 {'generated_text': 'It can be seen that in 2014 the number of manufactured goods was higher than in 2015, but in 2015 the number of manufactured goods was less than in 2015. In contrast, in 2015 the number of manufactured goods increased by 8%. In 2015 the number'},
 {'generated_text': 'It can be seen that in 1999 the unemployment rate in Italy was the same as in the UK. In contrast in the period from 2000 to 2012 the unemployment rate in Italy was the sa

In [16]:
generator("It can be seen", num_return_sequences=10, return_full_text=True, do_sample=True, temperature=1)

[{'generated_text': 'It can be seen that in Africa the level of unemployment remained downward in 2014 and 2015 while the share of women and male workers was slightly higher. In Australia the level of unemployment remained stable but the share of men was level higher. In China one of'},
 {'generated_text': 'It can be seen that, in 1980 in Italy, the percentage of overweight men and girls in various categories in Italy are equal between the overweight, boys and girls - in 2008 the ratio of boys is 12 percent. In 1980, in Italy the same'},
 {'generated_text': 'It can be seen that from 2008 through 2012 the year the number of children are growing dramatically. Yemen became the dominant child and the youngest child in 2014. In the same year the number of boys and girls is rising. In 2014 the number of boys'},
 {'generated_text': 'It can be seen that in 2013 the number of men and women was raised in Europe. The percentage of women decreased in other world. There are two countries in the sec

In [4]:
gen_list = generator("It can be seen", num_return_sequences=10, return_full_text=True, do_sample=True, temperature=1)

In [5]:
nlp = spacy.load("en_core_web_sm")

We create a function aimed at generation moderation. It will filter normal sentences only selecting those ending with a dot and containing a year; it will replace any past tense verb forms it finds with present tense verb forms. SpaCy might miss some, but that is tolerable. We choose the 3rd person agreement as it is the most likely in this type of texts. The function will only print sentences with at least one error span present.

Before function is applied:

In [51]:
for i in gen_list:
    print(i['generated_text'])

It can be seen in 2013 when only 11%. in Yemen was 5 years old. In contrast to Japan the biggest numbers, but the rest of the USA was the same year from 1940 till 2000 and in USA the whole number was 7 years. In
It can be seen that between 1960 to 2012 unemployment in the USA and at the same time in Indonesia decreased, both the girls and boys are in the same stage. The same trend was rising a little. As in Sweden in 2013 unemployment declined. In
It can be seen, from 2001 to 2005 the amount of girls and boys became slightly smaller, while in 2006 the figures of boys and girls decreased, however by 2011, the numbers of girls and boys become more, it is significant, since the quantity
It can be seen the most dramatic changes in the 1990s from Yemen till 2006 in quantity of men over 30%. However, Sweden has the lowest trend for percentage with the same point of increase for the year in 2000, with the increase of the number
It can be seen that during the period from 1990 to 2014 Indonesia

After the function is applied:

In [50]:
for i in gen_list:
    doc = nlp(i['generated_text'])
    for j in doc.sents:
        flag = False
        j_text = j.text
        if j.text[-1] == '.':
            for token in j:
                if token.tag_  == 'VBD':
                    j_text = j_text.replace(token.text, '**'+token._.inflect("VBZ")+'**')
                if token.tag_ == 'VBZ' or token.tag_ == 'VBP':
                    j_text = j_text.replace(token.text, '**'+token.text+'**')
                if token.pos_ == 'NUM' and len(token.text) == 4:
                    flag = True
            if flag == True and j_text.find('**') != -1:
                print(j_text)
    print('-----------')

In contrast to Japan the biggest numbers, but the rest of the USA **is** the same year from 1940 till 2000 and in USA the whole number **is** 7 years.
-----------
It can be seen that between 1960 to 2012 unemployment in the USA and at the same time in Indonesia **decreases**, both the girls and boys **are** in the same stage.
As in Sweden in 2013 unemployment **declines**.
-----------
-----------
-----------
It can be seen that during the period from 1990 to 2014 Indonesia **has** a decrease, approximately twice as big as the two world number.
-----------
It can be seen that in 1960 in Japan population of males and females **increases** about 3%.
In 2000 the increase of th**is** level ****is**** slightly more then 4% and it **is** about 3%.
-----------
In Yemen the decrease gradually **rise** faster, from 11 percents in 2000 to 6 percents in 2012 and 7 percents in 2013.
-----------
It can be seen that in 2012 the number of men and women aged 65 and older in the USA **starts** a signifi